In [5]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[0]
sys.path.append(str(project_root))

In [6]:
from modules.lizard import create_lizard_attention_config

In [7]:
import torch

In [8]:
cfg = create_lizard_attention_config(
    batch_size=2,
    seq_len=8192 * 4,
    hidden_size=256,
    num_heads=4,
    dtype="float32",
    device="cuda" if torch.cuda.is_available() else "cpu",
)

In [9]:
from modules.lizard_attention_block_triton import LizardAttentionBlock

In [10]:
torch_dtype = cfg.dtype.to_torch()
head_dim = cfg.hidden_size // cfg.num_heads

print(f"Configuring model with: {cfg}")
print(f"Head Dimension: {head_dim}")

model = (
    LizardAttentionBlock(
        d_model=cfg.hidden_size, n_heads=cfg.num_heads, window_size=32, alpha=0.5, m=4
    )
    .to(cfg.device)
    .to(torch_dtype)
)

q = torch.randn(
    cfg.batch_size,
    cfg.num_heads,
    cfg.seq_len,
    head_dim,
    device=cfg.device,
    dtype=torch_dtype,
)
k = torch.randn(
    cfg.batch_size,
    cfg.num_heads,
    cfg.seq_len,
    head_dim,
    device=cfg.device,
    dtype=torch_dtype,
)
v = torch.randn(
    cfg.batch_size,
    cfg.num_heads,
    cfg.seq_len,
    head_dim,
    device=cfg.device,
    dtype=torch_dtype,
)

output = model(q, k, v)

print(f"Input shape:  {q.shape}")
print(f"Output shape: {output.shape}")

assert q.shape == output.shape

Configuring model with: LizardAttentionBlockConfig(batch_size=2, seq_len=32768, hidden_size=256, num_heads=4, dtype=DTypeConfig(name='float32'), device='cuda')
Head Dimension: 64
Input shape:  torch.Size([2, 4, 32768, 64])
Output shape: torch.Size([2, 4, 32768, 64])


In [11]:
# 1. Warmup (Crucial!)
# Run the model a few times to compile kernels and populate caches.
# If you skip this, your first measurement will be 10x-100x slower.
print("Warming up...")
for _ in range(10):
    _ = model(q, k, v)
torch.cuda.synchronize()

# 2. Benchmark Loop
start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)
timings = []

print("Benchmarking...")
with torch.no_grad(): # Disable gradients for pure inference speed
    for _ in range(100): # Run 100 times for statistical stability
        start_event.record()
        _ = model(q, k, v)
        end_event.record()
        
        # Wait for this specific run to finish
        torch.cuda.synchronize()
        timings.append(start_event.elapsed_time(end_event))

# 3. Report
import numpy as np
mean_time = np.mean(timings)
std_time = np.std(timings)

print(f"Average latency: {mean_time:.3f} ms ± {std_time:.3f} ms")

Warming up...
Benchmarking...
Average latency: 56.243 ms ± 0.555 ms


In [12]:
!pip install seaborn

Defaulting to user installation because normal site-packages is not writeable


In [13]:
import torch
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Import your classes
from modules.lizard_attention_block_pytorch import LizardAttentionBlock as PyTorchBlock
from modules.lizard_attention_block_triton import LizardAttentionBlock as TritonBlock

def benchmark_scaling():
    device = "cuda"
    dtype = torch.float16
    D = 64
    H = 4
    
    # Define lengths to test: small -> medium -> large -> huge
    lengths = [128, 512, 2048, 4096, 8192, 16384, 16384 * 2]
    results = []

    print(f"{'Length':<10} | {'PyTorch (ms)':<15} | {'Triton (ms)':<15} | {'Speedup':<10}")
    print("-" * 60)

    for L in lengths:
        # 1. Setup Models
        m_pt = PyTorchBlock(d_model=D*H, n_heads=H).to(device).to(dtype)
        m_tr = TritonBlock(d_model=D*H, n_heads=H).to(device).to(dtype)
        
        # 2. Setup Data
        q = torch.randn(1, H, L, D, device=device, dtype=dtype)
        k = torch.randn(1, H, L, D, device=device, dtype=dtype)
        v = torch.randn(1, H, L, D, device=device, dtype=dtype)
        
        # 3. Warmup both
        for _ in range(5):
            m_pt(q, k, v)
            m_tr(q, k, v)
        torch.cuda.synchronize()

        # 4. Measure PyTorch
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        
        start.record()
        for _ in range(20): m_pt(q, k, v)
        end.record()
        torch.cuda.synchronize()
        pt_time = start.elapsed_time(end) / 20

        # 5. Measure Triton
        start.record()
        for _ in range(20): m_tr(q, k, v)
        end.record()
        torch.cuda.synchronize()
        tr_time = start.elapsed_time(end) / 20
        
        speedup = pt_time / tr_time
        print(f"{L:<10} | {pt_time:.4f}          | {tr_time:.4f}          | {speedup:.2f}x")
        
        results.append({"L": L, "Method": "PyTorch", "Time": pt_time})
        results.append({"L": L, "Method": "Triton", "Time": tr_time})

benchmark_scaling()

Length     | PyTorch (ms)    | Triton (ms)     | Speedup   
------------------------------------------------------------
128        | 1.2363          | 0.2991          | 4.13x
512        | 1.2274          | 0.6622          | 1.85x
2048       | 3.2911          | 3.1411          | 1.05x
4096       | 6.4688          | 6.1973          | 1.04x
8192       | 13.0915          | 12.2083          | 1.07x
16384      | 26.5550          | 28.8718          | 0.92x
32768      | 59.4245          | 59.2301          | 1.00x
